# LAYA v7.1 — Train Run #1 + Backtest (Google Colab, free T4)

**What this notebook does, top to bottom:** verifies the frozen validation split, smoke-tests the integrated runtime (rule-based, CPU), compares the 3 candidate backbones, trains the v7.1 classifier on a nested learning curve (2,000 → 4,000 → 7,500), calibrates graph-aware OOF thresholds, and backtests the trained checkpoint on the frozen validation split (500), Kaanoon-28, the 16-Q regression set, and Brutal-230 — through the integrated runtime (contract C1) and the backtest harness (contract C3).

**Colab-only rule (user instruction):** all computing / testing / code runs on Google Colab. No Kaggle, no comparison. The RTX 5050 commands in `docs/rtx-recall-training-commands.md` are the equivalent local path, not a comparison.

**Honest claim rule:** no claim of ≥70% exact is valid until a sealed re-run measures it (plan `docs/70-percent-plan.md` §6, Phase E). The frozen-split and Kaanoon numbers this notebook prints are proxies, not proof. This notebook contains **no sealed re-run cell** — sealed runs happen only after the Phase D gates pass, in a separate session.

**How to run:** open this notebook in Colab → Runtime → Change runtime type → **T4 GPU** → set `CPU_SMOKE_TEST = False` in the first code cell → Runtime → Run all. With `CPU_SMOKE_TEST = True` (the default) only the CPU smoke path runs: no GPU, no training, no torch — that is the path verifiable on a CPU-only VM before any GPU run.


In [ ]:
# === LAYA v7.1 — Train Run #1 + backtest (Colab T4) ===
# CPU_SMOKE_TEST = True  -> CPU-only path: verifies the integrated runtime
#   end to end with the RULE-BASED extractor. No GPU, no training, no torch.
#   This is the path verifiable on a CPU-only VM. Set False for the GPU run.
# CPU_SMOKE_TEST = False -> full GPU run: clone, pip install, 3-backbone
#   smoke test, nested-curve training, OOF calibration, backtest.
CPU_SMOKE_TEST = True

import hashlib
import json
import subprocess
import sys
from pathlib import Path

EXPECTED_VAL_SHA256 = "7640d067c28abf539131dbcc891e4bdffedf62bbef4a7d66c1659bb96d07e58e"
REPO_URL = "https://github.com/Knnivedh/laya-family-law"


def sh(cmd, check=False):
    """Run a shell command from the repo root; print it and its exit code. Never raises."""
    print(f"$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True, cwd=str(REPO_ROOT))
    print(f"[exit {r.returncode}]", flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r.returncode


In [ ]:
# Resolve the repo root and print the exact HEAD this run is executing.
# In GPU mode the repo is cloned fresh below (first); in CPU-smoke mode the
# repo already on disk is used. A stale checkout is visible here — check the
# HEAD printed against the push the coordinator announced before trusting
# any number below.
import os

if not CPU_SMOKE_TEST and not Path("/content/laya-family-law").exists():
    print("Cloning", REPO_URL, "…")
    subprocess.run(["git", "clone", REPO_URL, "/content/laya-family-law"], check=True)
    os.chdir("/content/laya-family-law")
elif not CPU_SMOKE_TEST:
    os.chdir("/content/laya-family-law")
    print("Repo already cloned — pulling latest main:")
    subprocess.run(["git", "-C", "/content/laya-family-law", "checkout", "main"])
    subprocess.run(["git", "-C", "/content/laya-family-law", "pull", "--ff-only"])

_CANDIDATES = [Path.cwd(), Path("/content/laya-family-law"),
               Path.home() / "workspace" / "laya-v71-work"]
REPO_ROOT = next((p for p in _CANDIDATES if (p / "src" / "laya_integrated.py").exists()), Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
print("REPO_ROOT:", REPO_ROOT)
print("HEAD:", subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
      capture_output=True, text=True).stdout.strip())
print("git status (short):")
sh("git status --short")


## ⚠️ PUSH DEPENDENCY — read before running the GPU cells

The new implementation files this notebook calls are **NOT on GitHub yet** — they live only in the working copy `~/workspace/laya-v71-work` until the parent (coordinator) verifies and pushes them:

- `src/laya_integrated.py` (Lane C, contract C1 — already committed locally, `792c75f`)
- `tests/integrated_backtest.py` (Lane E, contract C3 — **landed in the working copy, uncommitted**, while this notebook was being written; verified `--help` + a full rule-based run on this VM, see `docs/impl_laneD_notebook.md`)
- `src/laya_v7_classifier.py` + `scripts/train_v71_classifier.py` (Lane B amendments — **uncommitted edits in progress** when this notebook was written; training commands below are grounded in their current tree state)
- `scripts/smoke_backbones.py` (Lane A — **landed in the working copy, uncommitted**, while this notebook was being written; `--help` verified)
- `scripts/calibrate_thresholds.py` (Lane B — **landed in the working copy, uncommitted**, while this notebook was being written; `--help` verified, real CLI used below)

**Do not run the GPU cells until the coordinator confirms the push has landed.** The notebook prints the repo HEAD it is running from (cell above) — if that HEAD does not contain the files above, the GPU cells print a loud `SKIP … has not landed yet` instead of failing silently, and the backtest/training steps must wait for the push. The CPU smoke path (cells tagged `cpu_smoke`) does not need the push: it only needs `src/laya_integrated.py` and the frozen ground-truth file, both already in the working copy.


In [ ]:
# GPU check — needs no torch, only the nvidia-smi binary.
if CPU_SMOKE_TEST:
    print("SKIP (CPU_SMOKE_TEST=True): no GPU needed for the smoke path.")
else:
    sh("nvidia-smi")


In [ ]:
# Install only what the training / smoke code imports:
#   torch + transformers (train script imports both lazily; the classifier
#   builds the model through transformers AutoTokenizer / AutoModel).
#   sentencepiece: the XLM-R slow-tokenizer path needs it in the 3-backbone
#   smoke test. No other packages are needed — eval F1 is computed by hand
#   (no sklearn), LoRA (peft) is a fallback only, not installed here.
if CPU_SMOKE_TEST:
    print("SKIP (CPU_SMOKE_TEST=True): no torch install for the smoke path.")
else:
    sh("pip install -q torch transformers sentencepiece")
    sh("pip show torch transformers sentencepiece | grep -E '^(Name|Version):'")
    sh("python3 -c \"import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')\"")


In [ ]:
# Verify the FROZEN validation split before ANY training or scoring.
# The split must be exactly the 500 records the recall swarm froze.
val = REPO_ROOT / "data" / "synthetic_v7_1" / "validation.jsonl"
train = REPO_ROOT / "data" / "synthetic_v7_1" / "train.jsonl"
sha = hashlib.sha256(val.read_bytes()).hexdigest()
n_train = sum(1 for _ in train.open(encoding="utf-8"))
n_val = sum(1 for _ in val.open(encoding="utf-8"))
print(f"train={n_train} val={n_val}")
print("validation sha256:", sha)
assert sha == EXPECTED_VAL_SHA256, "FROZEN SPLIT MISMATCH — stop, do not train or score."
print("OK: frozen validation split matches the frozen sha256. Training may proceed.")


In [ ]:
# CPU SMOKE TEST — runs in BOTH modes (rule-based runtime needs no GPU
# and must work before any training is attempted).
print("== integrated runtime --self-test (rule-based, no torch) ==")
sh("python3 src/laya_integrated.py --self-test")
print()
print("== 16-Q spot check, first 5 questions (rule-based; full 16-Q runs in the harness cell) ==")
from laya_integrated import run_question

gt = json.loads((REPO_ROOT / "ground-truth" / "ground-truth-v2-hma27-A.json").read_text(encoding="utf-8"))
exact = 0
for q in gt["questions"][:5]:
    r = run_question(q["question"], mode="rule-based")
    ok = set(r["sections"]) == set(q["gt_ids"])
    exact += ok
    print(f"{q['id']}: exact={ok} extractor={r['extractor']} sections={r['sections']} expected={sorted(q['gt_ids'])}")
print(f"16-Q spot check: {exact}/5 exact (full-set baseline is 14/16; the two known misses are H6 and H14)")


## 3-backbone smoke test (Lane A, contract C4)

Compares the three candidate backbones on the same smoke task, stratified by language (en / kn / kanglish). **No predetermined winner** — mmBERT-base is the primary candidate, MuRIL is a serious Kanglish challenger, XLM-R is the third leg:

- `jhu-clsp/mmBERT-base` (MIT)
- `google/muril-base-cased` (Apache-2.0)
- `FacebookAI/xlm-roberta-base` (MIT)

If the script has not landed yet this cell prints `SKIP` and the run continues to training with mmBERT-base first (the task spec's order) — the smoke result is then still owed from Lane A before any backbone is declared the winner.


In [ ]:
if CPU_SMOKE_TEST:
    print("SKIP (CPU_SMOKE_TEST=True): backbone smoke test needs a GPU.")
else:
    smoke = REPO_ROOT / "scripts" / "smoke_backbones.py"
    if not smoke.exists():
        print("SKIP: scripts/smoke_backbones.py is not in this checkout (Lane A, contract C4).")
        print("The notebook calls it by its contract path once the parent pushes Lane A's files.")
        print("Backbones to compare (no predetermined winner): jhu-clsp/mmBERT-base,")
        print("google/muril-base-cased, FacebookAI/xlm-roberta-base — stratified by en/kn/kanglish.")
    else:
        sh("python3 scripts/smoke_backbones.py --help")
        sh("python3 scripts/smoke_backbones.py")


## Train Run #1 — nested learning curve (Phase B)

Config (task spec + `scripts/train_v71_classifier.py` docstring, FALLBACK-2 as amended by the GPT re-run — this **supersedes** the pre-amendment Phase B config in `docs/70-percent-plan.md` §3, which still says micro-batch 1 / ASL / max_seq_len 288):

| Setting | Value |
|---|---|
| Backbone (first) | `jhu-clsp/mmBERT-base` |
| Loss | `weighted_ce` (baseline; `focal_ce` is the challenger, not run here) |
| Encoding | 1-pass span pooling (`--pooling one_pass`) |
| Max seq len / clauses | 320 / 8 |
| Precision | bf16 (`--bf16`) |
| Micro-batch / grad-accum | 8 / 2 (effective 16). **Target, not guaranteed** — if it OOMs the script FAILS loudly suggesting `--batch-size 4 --grad-accum 4`; it never silently shrinks the batch. Re-run that subset with those two flags only. |
| Epochs | max 4, early stopping `--patience 2` |
| LR | 2e-5 encoder / 1e-4 heads (`--lr` / `--head-lr`) |
| Schedule | linear warmup 10% → cosine decay (`--warmup-frac 0.1`, built into the script) |
| Weight decay | 0.01 |
| Seed | 42 |

Curve: `--train-subset 2000` → `models/v71-classifier-2k`, then `--train-subset 4000` → `models/v71-classifier-4k`, then the full 7,500 → `models/v71-classifier` (the default C2 checkpoint path the integrated runtime looks for: `models/v71-classifier/best/model.pt`). **Decision rule (plan §3):** if the curve has already flattened below target, adding more of the same data will not solve it — go to Phase C with LLM-diversified data, do not just add more templates.


In [ ]:
# Corpus load check — CPU-safe, no torch. Expected: OK: train=7500 val=500.
sh("python3 scripts/train_v71_classifier.py --train-file data/synthetic_v7_1/train.jsonl --val-file data/synthetic_v7_1/validation.jsonl --check-data-only")


In [ ]:
# Nested learning curve: 2,000 -> 4,000 -> 7,500. Each run writes its own
# models/<name>/best/model.pt + config.json + label_maps.json + train_log.jsonl.
# (thresholds.json is NOT written by training — it comes from the calibration cell.)
if CPU_SMOKE_TEST:
    print("SKIP (CPU_SMOKE_TEST=True): no training on the CPU smoke path.")
else:
    BASE = ("--train-file data/synthetic_v7_1/train.jsonl "
            "--val-file data/synthetic_v7_1/validation.jsonl "
            "--backbone jhu-clsp/mmBERT-base --epochs 4 --batch-size 8 --grad-accum 2 "
            "--lr 2e-5 --head-lr 1e-4 --loss weighted_ce --pooling one_pass "
            "--max-seq-len 320 --max-clauses 8 --weight-decay 0.01 --warmup-frac 0.1 "
            "--patience 2 --device cuda --bf16 --grad-checkpoint --seed 42")
    print("== subset 2000 -> models/v71-classifier-2k ==")
    sh(f"python3 scripts/train_v71_classifier.py {BASE} --train-subset 2000 --output-dir models/v71-classifier-2k")
    print("== subset 4000 -> models/v71-classifier-4k ==")
    sh(f"python3 scripts/train_v71_classifier.py {BASE} --train-subset 4000 --output-dir models/v71-classifier-4k")
    print("== full 7500 -> models/v71-classifier (default checkpoint path) ==")
    sh(f"python3 scripts/train_v71_classifier.py {BASE} --output-dir models/v71-classifier")


In [ ]:
# Score each subset's BEST checkpoint on the frozen validation split
# (per-head label metrics + section-exact, printed by the train script).
if CPU_SMOKE_TEST:
    print("SKIP (CPU_SMOKE_TEST=True): no checkpoints to evaluate on the smoke path.")
else:
    for name in ["v71-classifier-2k", "v71-classifier-4k", "v71-classifier"]:
        print(f"== eval {name} on frozen validation 500 ==")
        sh(f"python3 scripts/train_v71_classifier.py --eval-checkpoint models/{name}/best/model.pt "
           f"--val-file data/synthetic_v7_1/validation.jsonl --device cuda")


In [ ]:
# Learning-curve results table (from each run's train_log.jsonl).
# If a run has not happened yet the row reads NOT RUN — never invent a number.
if CPU_SMOKE_TEST:
    print("SKIP (CPU_SMOKE_TEST=True): no training runs to tabulate.")
else:
    print(f"{'subset':>7} | {'epochs logged':>13} | best mean val macro-F1 (issue+fact)")
    print("-" * 70)
    for subset, name in [(2000, "v71-classifier-2k"), (4000, "v71-classifier-4k"), (7500, "v71-classifier")]:
        log = REPO_ROOT / "models" / name / "train_log.jsonl"
        if not log.exists():
            print(f"{subset:>7} | NOT RUN        | (no train_log.jsonl — run the training cell first)")
            continue
        rows = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines() if line.strip()]
        best = None
        for r in rows:
            for k, v in r.items():
                if "mean_macro" in k and isinstance(v, (int, float)):
                    best = max(best, v) if best is not None else v
        print(f"{subset:>7} | {len(rows):>13} | {best if best is not None else '(see log keys: ' + ', '.join(rows[-1].keys()) + ')'}")
        print(f"        last epoch row: {rows[-1]}")


## Graph-aware OOF threshold calibration

- Dump per-record predictions on the **train split only** (`--val-file data/synthetic_v7_1/train.jsonl` + `--dump-predictions`). The calibration script **refuses validation files** — OOF is on Train-7500 ONLY, never Val-500, so the frozen split stays untouched by calibration.
- Calibrate with kappa in {50, 150, 300} (κ=150 is an unproven hyperparameter — compare all three, separate priors per head family, up to 3 coordinate passes), temperature scaling first, precision floor 0.91. Writes `models/v71-classifier/best/thresholds.json` next to the checkpoint.
- CLI verified against the landed script (`python3 scripts/calibrate_thresholds.py --help`, this VM, 2026-11-27): `--oof-predictions … --train-file … --checkpoint-dir … --kappas 50,150,300 --max-passes 3`. The script refuses validation files — OOF is on Train-7500 ONLY.


In [ ]:
if CPU_SMOKE_TEST:
    print("SKIP (CPU_SMOKE_TEST=True): calibration needs a trained checkpoint.")
else:
    print("== OOF prediction dump — TRAIN split only (never Val-500) ==")
    sh("python3 scripts/train_v71_classifier.py --eval-checkpoint models/v71-classifier/best/model.pt "
       "--val-file data/synthetic_v7_1/train.jsonl --device cuda "
       "--dump-predictions models/v71-classifier/oof-preds.jsonl")
    cal = REPO_ROOT / "scripts" / "calibrate_thresholds.py"
    if not cal.exists():
        print("SKIP: scripts/calibrate_thresholds.py is not in this checkout (Lane B).")
        print("OOF predictions are saved at models/v71-classifier/oof-preds.jsonl; run the")
        print("calibration once the script lands. Contract: OOF on Train-7500 ONLY,")
        print("kappa in {50, 150, 300}, up to 3 coordinate passes; writes")
        print("models/v71-classifier/best/thresholds.json. The backtest below still runs")
        print("(argmax decode, no calibrated thresholds).")
    else:
        sh("python3 scripts/calibrate_thresholds.py --help")
        sh("python3 scripts/calibrate_thresholds.py --oof-predictions models/v71-classifier/oof-preds.jsonl "
           "--train-file data/synthetic_v7_1/train.jsonl --checkpoint-dir models/v71-classifier/best "
           "--kappas 50,150,300 --max-passes 3")
        th = REPO_ROOT / "models" / "v71-classifier" / "best" / "thresholds.json"
        print("thresholds.json written:", th.exists())
        if th.exists():
            print(th.read_text(encoding="utf-8")[:2000])


## Backtest — integrated harness (contract C3)

Runs `tests/integrated_backtest.py --mode classifier --suite all` (frozen validation 500, Kaanoon-28, 16-Q, Brutal-230), then the same suites in `--mode rule-based` as the baseline comparison. The harness imports the integrated runtime (C1) — it never subprocesses it for scoring — prints per-set exact / micro-P / micro-R / silent + the frozen gate rating, and refuses any sealed suite.

**Gates to watch — rule-based references measured with THIS harness on this VM (2026-11-27, Lane D verification, `--mode rule-based --suite all`):** frozen validation **261/500 = 52.2%** (micro-P/R 0.8905/0.4623, silent 212); Kaanoon-28 **14/28 = 50.0%**; 16-Q **14/16** (must stay 14/16, no regression); Brutal-230 **210/230 = 91.3%** (micro-P/R 0.9974/0.9205, silent 15).

**⚠️ Brutal-230 discrepancy, recorded honestly:** the shared state lists Brutal-230 as "230/230", but that number is from the **v7 live run** (a different code path, RTX), not from the integrated runtime. Through this harness + integrated runtime the measured rule-based reference is **210/230** — the harness's own baselines block confirms it (pre-recall-swarm reference 215/230, current-code reference 210/230, measured 210/230 = MATCH). So the classifier gate through this notebook is: **do not regress below 210/230 on Brutal-230**, and closing the gap toward 230/230 is the aspiration, not the measured C1 baseline. Do not report a classifier Brutal number against 230/230 as if 230/230 were the C1 baseline. The coordinator owns reconciling the 230/230 (v7 live) vs 210/230 (C1 harness) gap.

A classifier result above the rule-based baseline on the frozen split is a proxy improvement only — the ≥70% claim still needs the sealed re-run (Phase E).


In [ ]:
if CPU_SMOKE_TEST:
    print("SKIP (CPU_SMOKE_TEST=True): backtest needs a trained checkpoint.")
else:
    sh("mkdir -p results")
    h = REPO_ROOT / "tests" / "integrated_backtest.py"
    if not h.exists():
        print("SKIP: tests/integrated_backtest.py is not in this checkout (Lane E, contract C3).")
        print("The notebook calls it by its contract path once the parent pushes Lane E's files:")
        print("  python3 tests/integrated_backtest.py --mode classifier --suite all --json-out results/laya_v71_backtest_classifier.json")
        print("Expected per-set output: exact / micro-P / micro-R / silent + frozen gate rating,")
        print("for frozen-500, Kaanoon-28, 16-Q (must stay 14/16), Brutal-230 (must stay 210/230 -- the C1 rule-based reference; 230/230 is the v7-live aspiration).")
    else:
        print("== backtest: classifier mode, all suites ==")
        sh("python3 tests/integrated_backtest.py --mode classifier --suite all "
           "--checkpoint models/v71-classifier/best/model.pt --json-out results/laya_v71_backtest_classifier.json")
        print("== backtest: rule-based mode, all suites (baseline comparison) ==")
        sh("python3 tests/integrated_backtest.py --mode rule-based --suite all --json-out results/laya_v71_backtest_rule_based.json")
        print()
        print("HONEST CLAIM RULE: no >=70% claim until a sealed re-run measures it (Phase E,")
        print("docs/70-percent-plan.md §6). Frozen-split / Kaanoon numbers above are proxies, not proof.")


## Phase E — sealed re-run: NOT in this notebook (deliberately)

There is **no sealed re-run cell** here. The sealed sets (`test-bank/sealed/sealed-fresh-20.json`, `test-bank/sealed/sealed-kannada-8.json`) are never loaded by this notebook, and no cell references them. The sealed re-run happens only after the Phase D gates pass (300-item independent human-written holdout, ≥80% per language), in a separate session, on a frozen checkpoint — targets 14/20 Fresh-20 and 6/8 Kannada-8. The disclosure rule applies when reporting it: the aggregate sealed results have already informed development priorities, the sealed sets are too small to establish a population-level 70% guarantee, and neither sealed set establishes Kanglish accuracy.


In [ ]:
# Run summary — what exists on disk now, and what is still open.
print("LAYA v7.1 Train Run #1 — run summary")
print("REPO_ROOT:", REPO_ROOT)
print("HEAD:", subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
      capture_output=True, text=True).stdout.strip())
print("CPU_SMOKE_TEST:", CPU_SMOKE_TEST)
print()
for label, path in [
    ("checkpoint 2k", "models/v71-classifier-2k/best/model.pt"),
    ("checkpoint 4k", "models/v71-classifier-4k/best/model.pt"),
    ("checkpoint 7.5k (default)", "models/v71-classifier/best/model.pt"),
    ("thresholds.json", "models/v71-classifier/best/thresholds.json"),
    ("OOF predictions", "models/v71-classifier/oof-preds.jsonl"),
    ("backtest classifier JSON", "results/laya_v71_backtest_classifier.json"),
    ("backtest rule-based JSON", "results/laya_v71_backtest_rule_based.json"),
]:
    p = REPO_ROOT / path
    print(f"[{'OK ' if p.exists() else ' -- '}] {label}: {path}")
print()
print("Still open (see docs/impl_laneD_notebook.md): the Lane A / B / E files this")
print("notebook calls landed in the working copy but are NOT on GitHub yet — the GPU")
print("cells need the coordinator's push first. Brutal-230: the C1 harness rule-based")
print("reference is 210/230, not the 230/230 v7-live number — see the backtest cell.")
print("The sealed re-run is a separate Phase E session after Phase D.")
print()
print("HONEST CLAIM RULE: no >=70% claim until a sealed re-run measures it.")
